# Building full-stack applications

**Optional depth track · module 1 of 5**

**Goal:** A CLI is not an application. Put a boundary in front of the agent you already built, and decide what a caller sees when it works, when it refuses, and when it breaks.

**Why it matters:** Your coding agent will happily generate an endpoint. It will not decide for you whether the trace goes in the response, whether a refusal is a 200 or a 422, or what a 500 is allowed to say. Those three calls are the whole exercise, and each one is a place a generated API leaks something or lies to its caller.

Nothing here is graded and nothing in the fifteen sessions depends on it. Work through it when you want the layer underneath.

In [ ]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
    print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    preflight(REPO_ROOT)

In [ ]:
# depth_checks registers this track's checkers. check() and review() are the
# same ones the course uses.
from bootcamp_agent.checks import check, review
import bootcamp_agent.depth_checks  # noqa: F401

## 1. The response contract

**Context.** `answer_question` returns an `AgentResult` with a typed answer and a trace.
The trace is the most useful thing in it for you, and it contains the prompt, the retrieved
passages and the tool arguments. Decide what crosses the boundary.

**Instructions.**

1. Fill `body` with the response a caller gets for a successful, cited answer.
2. Include a `request_id`. Without one you cannot follow a single call through a log.
3. Leave out anything a caller does not need. The check refuses a body that carries the trace.

In [ ]:
import json

body = {
    "answer": "Chunking splits a document into passages the retriever can score.",
    "citations": ["rag-basics"],
    "confidence": 0.8,
    "needs_human_review": False,
    "request_id": "req_01J7E8Q2M4",
}
print(json.dumps(body, indent=2))

**Expected output**

```
{
  "answer": "Chunking splits a document into passages the retriever can score.",
  ...
  "needs_human_review": false,
  "request_id": "req_01J7E8Q2M4"
}
✅ d1-e1 passed
```

In [ ]:
check("d1-e1", body)

## 2. The error envelope

**Context.** Two failures, two audiences. A caller who sent a bad request can fix it, so tell
them exactly what was wrong. A caller who hit an internal error cannot fix anything, and the
exception text is where tokens, file paths and table names escape.

**Instructions.**

1. `error_for(kind, detail)` returns `{"error": {"code", "message", "request_id"}}`.
2. For `invalid_request`, the message names the offending field.
3. For `internal`, the message must not contain `detail`. Write a sentence a support agent
   can read out loud.

In [ ]:
def error_for(kind: str, detail: str) -> dict:
    request_id = "req_01J7E8Q2M4"  # in real life, per request
    if kind == "invalid_request":
        return {"error": {"code": "invalid_request", "message": detail, "request_id": request_id}}
    return {
        "error": {
            "code": "internal",
            "message": "Something failed on our side. Quote the request id when you report it.",
            "request_id": request_id,
        }
    }


for kind, detail in [("invalid_request", "question must be a string"),
                     ("internal", "KeyError: 'secret_token' at agent.py:88")]:
    print(kind, "->", error_for(kind, detail))

**Expected output**

```
invalid_request -> {'error': {'code': 'invalid_request', 'message': 'question must be a string', ...}}
internal -> {'error': {'code': 'internal', 'message': 'Something failed on our side. ...', ...}}
✅ d1-e2 passed
```

In [ ]:
check("d1-e2", error_for)

## 3. What the client has to show

**Context.** A refusal is a **successful** response. The agent did its job: it found no
grounding and said so. If the client renders that in the same red box as a 500, users learn
to distrust a working system.

**Instructions.**

1. Name every state the client moves through, and say what the user actually sees in each.
2. `refused` and `error` must both be there, and they must not look the same.
3. Four states minimum: something before, something during, and both outcomes.

In [ ]:
states = {
    "idle": "An empty box with the question field focused and an example question below it.",
    "loading": "The button becomes a spinner and the field locks, so nobody sends it twice.",
    "answered": "The answer, with each citation as a link to the document it came from.",
    "refused": "A plain note saying the corpus does not cover this, with the question kept "
               "so it can be reworded. Not styled as a failure.",
    "error": "A short apology, the request id, and a retry button. The question is kept.",
}
for name, seen in states.items():
    print(f"{name:10} {seen}")

**Expected output**

```
idle       An empty box with the question field focused ...
loading    The button becomes a spinner ...
answered   The answer, with each citation as a link ...
refused    A plain note saying the corpus does not cover this ...
error      A short apology, the request id, and a retry button ...
✅ d1-e3 passed
```

In [ ]:
check("d1-e3", states)

## Review

The scorecard for this module. Every ❌ names the exercise and the hint.

In [ ]:
review("d1")